### Using Date not timestamp

In [3]:
# 🔄 ETL: PostgreSQL → Doris Bronze Layer
# **Pattern**: Extract → Transform (add metadata) → Load  
# **API**: Spark DataFrame (recommended over RDD)

# %%
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, lit, to_timestamp
from pyspark.sql.types import StructType, StructField, LongType, StringType, DecimalType, DateType, TimestampType

# ============================================
# 🔧 Configuration (Load from .env or defaults)
# ============================================
def load_config():
    """Load connection config from environment (mounted .env)"""
    return {
        "pg": {
            "url": f"jdbc:postgresql://{os.getenv('POSTGRES_HOST', 'host.docker.internal')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DB', 'source_db')}",
            "user": os.getenv("POSTGRES_USER", "etl_user"),
            "password": os.getenv("POSTGRES_PASSWORD", ""),
            "driver": "org.postgresql.Driver",
            "table": "public.products",
            "partition_column": "id",
            "lower_bound": os.getenv("PG_ID_MIN", "1"),
            "upper_bound": os.getenv("PG_ID_MAX", "1000000"),
            "num_partitions": int(os.getenv("PG_NUM_PARTITIONS", "4")),
        },
        "doris": {
            "fenodes": f"{os.getenv('DORIS_FE_HOST', 'host.docker.internal')}:{os.getenv('DORIS_FE_HTTP_PORT', '8030')}",
            "table": "bronze.products",
            "user": os.getenv("DORIS_USER", "root"),
            "password": os.getenv("DORIS_PASSWORD", ""),
            "format": "json",
            "batch_size": int(os.getenv("DORIS_BATCH_SIZE", "50000")),
            "enable_2pc": os.getenv("DORIS_ENABLE_2PC", "true").lower() == "true",
        },
        "spark": {
            "app_name": "postgres_to_doris_bronze",
            "master": os.getenv("SPARK_MASTER", "spark://spark-master:7077"),
            "jars": "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar",
        }
    }

cfg = load_config()
print(f"🔧 Config: PostgreSQL={cfg['pg']['table']}, Doris={cfg['doris']['table']}")

# ============================================
# Step 1: Initialize Spark Session
# ============================================
print("🚀 Initializing Spark Session...")
spark = SparkSession.builder \
    .appName(cfg["spark"]["app_name"]) \
    .master(cfg["spark"]["master"]) \
    .config("spark.jars", cfg["spark"]["jars"]) \
    .config("spark.driver.host", "jupyter-lab") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark {spark.version} @ {spark.sparkContext.master}")

# ============================================
# Step 2: EXTRACT - Read from PostgreSQL
# ============================================
print("\n📥 EXTRACT: Reading from PostgreSQL...")

pg_df = spark.read \
    .format("jdbc") \
    .option("url", cfg["pg"]["url"]) \
    .option("dbtable", cfg["pg"]["table"]) \
    .option("user", cfg["pg"]["user"]) \
    .option("password", cfg["pg"]["password"]) \
    .option("driver", cfg["pg"]["driver"]) \
    .option("partitionColumn", cfg["pg"]["partition_column"]) \
    .option("lowerBound", cfg["pg"]["lower_bound"]) \
    .option("upperBound", cfg["pg"]["upper_bound"]) \
    .option("numPartitions", cfg["pg"]["num_partitions"]) \
    .option("fetchsize", "10000") \
    .load()

print(f"✅ Extracted {pg_df.count()} rows from PostgreSQL")
pg_df.printSchema()
pg_df.show(3, truncate=False)

# ============================================
# Step 3: TRANSFORM - Add Bronze Layer Metadata (DATE type)
# ============================================
print("\n🔄 TRANSFORM: Adding DWH metadata columns (DATE type)...")

batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{spark.sparkContext.applicationId}"

# ✅ FIXED: No comments between line continuations
bronze_df = pg_df \
    .withColumn("dwh_load_date", current_date()) \
    .withColumn("dwh_batch_id", lit(batch_id)) \
    .withColumn("dwh_source_system", lit("postgres_source")) \
    .withColumn("id", col("id").cast("bigint")) \
    .withColumn("description", col("description").cast("string")) \
    .withColumn("price", col("price").cast("decimal(10,2)")) \
    .withColumn("created_at", to_timestamp(col("created_at"))) \
    .select(
        col("id"),
        col("name"),
        col("description"),
        col("price"),
        col("created_at"),
        col("dwh_load_date"),
        col("dwh_batch_id"),
        col("dwh_source_system")
    )

print(f"✅ Transformed schema:")
bronze_df.printSchema()
print(f"📊 Sample transformed row:")
bronze_df.show(1, truncate=False)

# ============================================
# Step 4: LOAD - Write to Apache Doris
# ============================================
print("\n📤 LOAD: Writing to Apache Doris (bronze.products)...")

try:
    bronze_df.write \
        .format("doris") \
        .option("doris.table.identifier", cfg["doris"]["table"]) \
        .option("doris.fenodes", cfg["doris"]["fenodes"]) \
        .option("user", cfg["doris"]["user"]) \
        .option("password", cfg["doris"]["password"]) \
        .option("doris.sink.properties.format", cfg["doris"]["format"]) \
        .option("doris.sink.batch.size", cfg["doris"]["batch_size"]) \
        .option("doris.sink.enable-2pc", str(cfg["doris"]["enable_2pc"]).lower()) \
        .option("doris.write.fields", "id,name,description,price,created_at,dwh_load_date,dwh_batch_id,dwh_source_system") \
        .mode("append") \
        .save()
    
    print(f"✅ Successfully loaded {bronze_df.count()} rows to Doris bronze.products")
    print(f"🏷️  Batch ID: {batch_id}")
    print(f"📅 Load Date: {datetime.now().strftime('%Y-%m-%d')}")
    
except Exception as e:
    print(f"❌ Load failed: {type(e).__name__}: {str(e)[:300]}")
    print("💡 Troubleshooting:")
    print("   1. Verify Doris table: DESC bronze.products;")
    print("   2. Check dwh_load_date is DATE type (not DATETIME)")
    print("   3. Ensure column order matches in doris.write.fields")
    raise

# ============================================
# Step 5: VERIFY - Quick Validation
# ============================================
print("\n🔍 VERIFY: Reading back from Doris...")

try:
    verify_df = spark.read \
        .format("doris") \
        .option("doris.table.identifier", cfg["doris"]["table"]) \
        .option("doris.fenodes", cfg["doris"]["fenodes"]) \
        .option("user", cfg["doris"]["user"]) \
        .option("password", cfg["doris"]["password"]) \
        .option("doris.filter.query", f"dwh_load_date = '{datetime.now().strftime('%Y-%m-%d')}'") \
        .load()
    
    count = verify_df.count()
    print(f"✅ Verified: {count} rows loaded with dwh_load_date='{datetime.now().strftime('%Y-%m-%d')}'")
    
    if count > 0:
        print("📊 Sample loaded data:")
        verify_df.select("id", "name", "price", "dwh_load_date", "dwh_batch_id") \
            .show(3, truncate=False)
    
except Exception as e:
    print(f"⚠️ Verification read failed (non-critical): {e}")

# ============================================
# Step 6: Cleanup
# ============================================
spark.stop()
print("\n✨ ETL pipeline completed successfully")
print(f"🎯 Bronze layer ready: bronze.products (DATE-based batch tracking)")

🔧 Config: PostgreSQL=public.products, Doris=bronze.products
🚀 Initializing Spark Session...
✅ Spark 3.5.0 @ spark://spark-master:7077

📥 EXTRACT: Reading from PostgreSQL...
✅ Extracted 1261140 rows from PostgreSQL
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- created_at: timestamp (nullable = true)

+----+-------------+-------------+-------+-------------------+
|id  |name         |description  |price  |created_at         |
+----+-------------+-------------+-------+-------------------+
|1127|Item_4b598739|74.99.213.35 |167.77 |2025-12-07 22:56:29|
|1128|Item_25459f1a|41.234.55.189|6665.57|2025-09-08 08:33:37|
|1129|Item_f229d0b0|16.58.128.146|6496.20|2025-12-20 02:32:50|
+----+-------------+-------------+-------+-------------------+
only showing top 3 rows


🔄 TRANSFORM: Adding DWH metadata columns (DATE type)...
✅ Transformed schema:
root
 |-- id: long (nullable = t